In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("cover_type.csv")

## Data Understanding

In [3]:
df.shape

(145890, 13)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 145890 entries, 0 to 145889
Data columns (total 13 columns):
 #   Column                              Non-Null Count   Dtype
---  ------                              --------------   -----
 0   Elevation                           145890 non-null  int64
 1   Aspect                              145890 non-null  int64
 2   Slope                               145890 non-null  int64
 3   Horizontal_Distance_To_Hydrology    145890 non-null  int64
 4   Vertical_Distance_To_Hydrology      145890 non-null  int64
 5   Horizontal_Distance_To_Roadways     145890 non-null  int64
 6   Hillshade_9am                       145890 non-null  int64
 7   Hillshade_Noon                      145890 non-null  int64
 8   Hillshade_3pm                       145890 non-null  int64
 9   Horizontal_Distance_To_Fire_Points  145890 non-null  int64
 10  Cover_Type                          145890 non-null  str  
 11  Wilderness_Area                     145890 non-null  int64
 12 

In [5]:
df.describe()

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,Wilderness_Area,Soil_Type
count,145890.000000,145890.000000,145890.000000,145890.000000,145890.000000,145890.000000,145890.000000,145890.000000,145890.000000,145890.000000,145890.000000,145890.000000
mean,2874.458949,141.127418,11.925574,251.824738,34.554322,3313.827541,217.368106,224.874748,139.788203,3044.958105,1.186593,23.364905
std,210.801279,107.719296,6.319326,192.473899,41.215776,1687.779953,21.590298,16.084851,31.311690,1761.882341,0.656571,8.215184
min,1863.000000,0.000000,0.000000,0.000000,-146.000000,0.000000,0.000000,99.000000,0.000000,0.000000,1.000000,1.000000
25%,2747.000000,54.000000,7.000000,95.000000,7.000000,1848.000000,207.000000,216.000000,121.000000,1608.000000,1.000000,12.000000
50%,2909.000000,108.000000,11.000000,212.000000,23.000000,3420.000000,222.000000,226.000000,140.000000,2713.000000,1.000000,29.000000
75%,3004.000000,217.000000,15.000000,362.000000,51.000000,4673.000000,232.000000,236.000000,159.000000,4478.000000,1.000000,29.000000
max,3849.000000,360.000000,61.000000,1343.000000,554.000000,7117.000000,254.000000,254.000000,248.000000,7173.000000,4.000000,40.000000


- No Duplicates
- No missing values

In [12]:
df['Cover_Type'].value_counts()

Cover_Type
Lodgepole Pine       103071
Spruce/Fir            31110
Aspen                  3069
Krummholz              2160
Ponderosa Pine         2160
Douglas-fir            2160
Cottonwood/Willow      2160
Name: count, dtype: int64

- Let's use Class-weighted tree-based model, bcoz it is land data, we can't create fake samples or land data

## Data Cleaning & Transformation

In [13]:
df.isnull().sum()

Elevation                             0
Aspect                                0
Slope                                 0
Horizontal_Distance_To_Hydrology      0
Vertical_Distance_To_Hydrology        0
Horizontal_Distance_To_Roadways       0
Hillshade_9am                         0
Hillshade_Noon                        0
Hillshade_3pm                         0
Horizontal_Distance_To_Fire_Points    0
Cover_Type                            0
Wilderness_Area                       0
Soil_Type                             0
dtype: int64

In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 145890 entries, 0 to 145889
Data columns (total 13 columns):
 #   Column                              Non-Null Count   Dtype
---  ------                              --------------   -----
 0   Elevation                           145890 non-null  int64
 1   Aspect                              145890 non-null  int64
 2   Slope                               145890 non-null  int64
 3   Horizontal_Distance_To_Hydrology    145890 non-null  int64
 4   Vertical_Distance_To_Hydrology      145890 non-null  int64
 5   Horizontal_Distance_To_Roadways     145890 non-null  int64
 6   Hillshade_9am                       145890 non-null  int64
 7   Hillshade_Noon                      145890 non-null  int64
 8   Hillshade_3pm                       145890 non-null  int64
 9   Horizontal_Distance_To_Fire_Points  145890 non-null  int64
 10  Cover_Type                          145890 non-null  str  
 11  Wilderness_Area                     145890 non-null  int64
 12 

In [22]:
df['Soil_Type'].unique()

array([29, 12, 30, 18, 16, 20, 24, 23, 40, 19,  8, 22, 39,  9, 38, 33, 31,
       32, 11, 10,  5, 28,  4,  1, 13,  2, 17,  3, 34,  6, 14, 37, 35, 36,
       21, 26, 27, 25,  7])

In [23]:
import pandas as pd

num_cols = [
    "Elevation",
    "Slope",
    "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways",
    "Horizontal_Distance_To_Fire_Points"
]

outlier_summary = {}

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary[col] = outliers

pd.Series(outlier_summary)


Elevation                             4448
Slope                                 3385
Horizontal_Distance_To_Hydrology      2473
Vertical_Distance_To_Hydrology        7123
Horizontal_Distance_To_Roadways          0
Horizontal_Distance_To_Fire_Points       0
dtype: int64

In [24]:
for col in num_cols:
    print(f"\n{col}")
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outlier_df = df[(df[col] < lower) | (df[col] > upper)]
    print(outlier_df["Cover_Type"].value_counts(normalize=True))



Elevation
Cover_Type
Cottonwood/Willow    0.449191
Ponderosa Pine       0.204137
Krummholz            0.172662
Douglas-fir          0.156924
Spruce/Fir           0.015288
Lodgepole Pine       0.001799
Name: proportion, dtype: float64

Slope
Cover_Type
Lodgepole Pine       0.430133
Ponderosa Pine       0.157164
Cottonwood/Willow    0.125849
Douglas-fir          0.094830
Aspen                0.082127
Spruce/Fir           0.072378
Krummholz            0.037518
Name: proportion, dtype: float64

Horizontal_Distance_To_Hydrology
Cover_Type
Lodgepole Pine    0.552366
Spruce/Fir        0.334816
Krummholz         0.098666
Aspen             0.014153
Name: proportion, dtype: float64

Vertical_Distance_To_Hydrology
Cover_Type
Lodgepole Pine       0.579531
Spruce/Fir           0.189527
Krummholz            0.066826
Ponderosa Pine       0.053208
Cottonwood/Willow    0.042959
Aspen                0.042819
Douglas-fir          0.025130
Name: proportion, dtype: float64

Horizontal_Distance_To_Roadways

In [25]:
from scipy.stats import zscore

z_scores = df[num_cols].apply(zscore)
outliers = (z_scores.abs() > 3).sum()
outliers

Elevation                             2120
Slope                                 1663
Horizontal_Distance_To_Hydrology      1400
Vertical_Distance_To_Hydrology        2897
Horizontal_Distance_To_Roadways          0
Horizontal_Distance_To_Fire_Points       0
dtype: int64

Outliers were analyzed using IQR and Z-score methods. Since extreme values corresponded to meaningful ecological conditions—especially for minority forest types—no rows were removed. Tree-based models were used to naturally handle these extremes.

In [26]:
import numpy as np

skew_cols = [
    "Horizontal_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways",
    "Horizontal_Distance_To_Fire_Points"
]

for col in skew_cols:
    df[col] = np.log1p(df[col])


In [28]:
from sklearn.preprocessing import PowerTransformer

pt = PowerTransformer(method="yeo-johnson")
df["Vertical_Distance_To_Hydrology"] = pt.fit_transform(
    df[["Vertical_Distance_To_Hydrology"]]
)
    

In [29]:
df[num_cols].skew().sort_values(ascending=False)

Slope                                 0.943937
Vertical_Distance_To_Hydrology        0.308290
Elevation                            -0.664497
Horizontal_Distance_To_Fire_Points   -1.114535
Horizontal_Distance_To_Roadways      -1.367370
Horizontal_Distance_To_Hydrology     -2.152802
dtype: float64

## Feature Engineering

In [32]:
## Hillshade interactions
df["Hillshade_Diff_Noon_9am"] = df["Hillshade_Noon"] - df["Hillshade_9am"]
df["Hillshade_Diff_3pm_Noon"] = df["Hillshade_3pm"] - df["Hillshade_Noon"]

In [33]:
# Captures remoteness vs water proximity
df["Hydrology_Road_Ratio"] = (
    df["Horizontal_Distance_To_Hydrology"] /
    (df["Horizontal_Distance_To_Roadways"] + 1)
)


In [34]:
[w for w in df.columns if "Wilderness" in w or "Soil" in w][:5]


['Wilderness_Area', 'Soil_Type']

In [36]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)


TypeError: OneHotEncoder.__init__() got an unexpected keyword argument 'sparse'